[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/HyperCoast/blob/main/docs/examples/search_workspace.ipynb)

# Search Result Workspaces

This notebook demonstrates reusable search workspaces. It uses mock STAC-like items so the workflow is reproducible without network access or Earthdata credentials.

In [ ]:
# %pip install hypercoast

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import hypercoast

## Create a workspace from search items

In [ ]:
items = [
    {
        "id": "PACE_001",
        "collection": "PACE_OCI_L2_AOP_NRT",
        "properties": {"datetime": "2024-04-01T18:00:00Z", "cloud": 12},
        "href": "https://example.com/PACE_001.nc",
    },
    {
        "id": "PACE_002",
        "collection": "PACE_OCI_L2_AOP_NRT",
        "properties": {"datetime": "2024-04-02T18:00:00Z", "cloud": 65},
        "href": "https://example.com/PACE_002.nc",
    },
]

workspace = hypercoast.SearchResult.from_result(
    items,
    sensor="pace",
    query={"count": 2, "temporal": "2024-04-01/2024-04-02"},
)
workspace.to_dataframe()

## Filter, save, and reload

In [ ]:
clear_items = workspace.filter(cloud=12)
clear_items.to_dataframe()

In [ ]:
with TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)
    json_path = tmpdir / "pace-search.json"
    csv_path = tmpdir / "pace-search.csv"

    workspace.to_json(json_path)
    workspace.to_csv(csv_path)
    reloaded = hypercoast.load_search_result(json_path)

    print(json_path)
    print(csv_path)
    display(reloaded.to_dataframe())

## Use with live searches

For live catalog searches, pass backend results into the same workspace class.

```python
items = hypercoast.search_sensor("pace", count=5)
workspace = hypercoast.SearchResult.from_result(items, sensor="pace")
workspace.to_json("pace-search.json")
```